In [1]:
import os
import GEOparse
import pandas as pd
import gc # Garbage collector to free up memory

print("[*] Loading GSE65682 into memory...")
gse = GEOparse.get_GEO(filepath="/workspace/data/raw/GSE65682_family.soft.gz")

# ---------------------------------------------------------
# 1. CLEAN THE CLINICAL DATA
# ---------------------------------------------------------
clinical_df = gse.phenotype_data
char_cols = [col for col in clinical_df.columns if 'characteristics' in col]
clean_clinical = clinical_df[char_cols].copy()

# Rename columns
clean_clinical.columns = [col.split('.')[-1] for col in clean_clinical.columns]

# Filter out healthy controls
clean_clinical = clean_clinical[clean_clinical['mortality_event_28days'] != 'NA']
clean_clinical = clean_clinical.dropna(subset=['mortality_event_28days'])
clean_clinical['mortality_event_28days'] = pd.to_numeric(clean_clinical['mortality_event_28days'])
clean_clinical['age'] = pd.to_numeric(clean_clinical['age'], errors='coerce')

processed_dir = "/workspace/data/processed"
clinical_file = os.path.join(processed_dir, "GSE65682_clinical_clean.csv")
clean_clinical.to_csv(clinical_file)

print(f"\n[+] Cleaned Clinical Data:")
print(f"    Total Septic Patients: {clean_clinical.shape[0]}")
print(f"    Survivors: {sum(clean_clinical['mortality_event_28days'] == 0)}")
print(f"    Non-Survivors: {sum(clean_clinical['mortality_event_28days'] == 1)}")

# Free up memory before the heavy lifting
del clinical_df
gc.collect()

# ---------------------------------------------------------
# 2. MEMORY-OPTIMIZED GENE EXTRACTION
# ---------------------------------------------------------
print("\n[*] Manually extracting gene expressions for our 479 targets (Memory Safe Mode)...")

expr_dict = {}
patient_ids = clean_clinical.index.tolist()

# Loop through ONLY our target patients
for i, gsm_id in enumerate(patient_ids):
    # Print a progress tracker every 100 patients so you know it isn't frozen
    if (i + 1) % 100 == 0:
        print(f"    Processing patient {i + 1} / {len(patient_ids)}...")
        
    # Extract the 'VALUE' column and set the Gene ID (ID_REF) as the index
    expr_dict[gsm_id] = gse.gsms[gsm_id].table.set_index('ID_REF')['VALUE']

print("[*] Combining columns into final matrix...")
# Concatenate the dictionary directly into a DataFrame (Highly memory efficient)
expr_df = pd.DataFrame(expr_dict)

expr_file = os.path.join(processed_dir, "GSE65682_expression.csv")
expr_df.to_csv(expr_file)

print(f"[+] Saved Expression Data: {expr_df.shape[0]} genes, {expr_df.shape[1]} patients -> {expr_file}")
print("\n=== PHASE 2: DATA ENGINEERING COMPLETE ===")

27-Apr-2026 17:13:58 INFO GEOparse - Parsing /workspace/data/raw/GSE65682_family.soft.gz: 
27-Apr-2026 17:13:58 DEBUG GEOparse - DATABASE: GeoMiame
27-Apr-2026 17:13:58 DEBUG GEOparse - SERIES: GSE65682
27-Apr-2026 17:13:58 DEBUG GEOparse - PLATFORM: GPL13667


[*] Loading GSE65682 into memory...


/opt/conda/lib/python3.10/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
27-Apr-2026 17:14:06 DEBUG GEOparse - SAMPLE: GSM1602801
27-Apr-2026 17:14:06 DEBUG GEOparse - SAMPLE: GSM1602802
27-Apr-2026 17:14:06 DEBUG GEOparse - SAMPLE: GSM1602803
27-Apr-2026 17:14:06 DEBUG GEOparse - SAMPLE: GSM1602804
27-Apr-2026 17:14:06 DEBUG GEOparse - SAMPLE: GSM1602805
27-Apr-2026 17:14:06 DEBUG GEOparse - SAMPLE: GSM1602806
27-Apr-2026 17:14:07 DEBUG GEOparse - SAMPLE: GSM1602807
27-Apr-2026 17:14:07 DEBUG GEOparse - SAMPLE: GSM1602808
27-Apr-2026 17:14:07 DEBUG GEOparse - SAMPLE: GSM1602809
27-Apr-2026 17:14:07 DEBUG GEOparse - SAMPLE: GSM1602810
27-Apr-2026 17:14:07 DEBUG GEOparse - SAMPLE: GSM1602811
27-Apr-2026 17:14:07 DEBUG GEOparse - SAMPLE: GSM1602812
27-Apr-2026 17:14:07 DEBUG GEOparse - SAMPLE: GSM1602813
27-Apr-2026 17:14:07 DEBUG GEOpa


[+] Cleaned Clinical Data:
    Total Septic Patients: 479
    Survivors: 365
    Non-Survivors: 114

[*] Manually extracting gene expressions for our 479 targets (Memory Safe Mode)...
    Processing patient 100 / 479...
    Processing patient 200 / 479...
    Processing patient 300 / 479...
    Processing patient 400 / 479...
[*] Combining columns into final matrix...
[+] Saved Expression Data: 24646 genes, 479 patients -> /workspace/data/processed/GSE65682_expression.csv

=== PHASE 2: DATA ENGINEERING COMPLETE ===


In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import os

print("[*] Loading Processed Data (This will take a moment)...")
processed_dir = "/workspace/data/processed"

# Load Clinical Data
clinical_df = pd.read_csv(os.path.join(processed_dir, "GSE65682_clinical_clean.csv"), index_col=0)

# Load Expression Data and Transpose it (so Patients are Rows, Genes are Columns)
expr_df = pd.read_csv(os.path.join(processed_dir, "GSE65682_expression.csv"), index_col=0).T

# Ensure the patients match perfectly
assert all(clinical_df.index == expr_df.index), "Patient IDs do not match!"

print(f"[+] Data loaded perfectly. {expr_df.shape[1]} genes ready for evaluation.")

# ---------------------------------------------------------
# 1. PREPARE THE MACHINE LEARNING TARGETS
# ---------------------------------------------------------
# X = Our genes, y = Our target (1 = Dead, 0 = Survived)
X = expr_df
y = clinical_df['mortality_event_28days']

# Calculate class imbalance (Survivors vs Non-Survivors)
# XGBoost uses this to pay extra attention to the minority class (the patients who died)
imbalance_ratio = sum(y == 0) / sum(y == 1)

# ---------------------------------------------------------
# 2. TRAIN XGBOOST TO RANK GENES
# ---------------------------------------------------------
print("\n[*] Training XGBoost to find the most predictive genes...")
model = xgb.XGBClassifier(
    n_estimators=200,          # Build 200 decision trees
    max_depth=4,               # Keep trees shallow to prevent overfitting
    scale_pos_weight=imbalance_ratio, # Force AI to care about the minority class
    random_state=42,           # For reproducibility
    n_jobs=-1                  # Use all CPU cores
)

model.fit(X, y)

# ---------------------------------------------------------
# 3. EXTRACT AND SAVE TOP GENES
# ---------------------------------------------------------
print("[*] Ranking features...")
# Get feature importances from the model
importances = model.feature_importances_

# Create a DataFrame of Genes and their Importance Score
gene_ranking = pd.DataFrame({
    'Gene_ID': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Grab the Top 100 Genes
top_100_genes = gene_ranking.head(100)

print("\n=== THE TOP 10 PREDICTIVE GENES ===")
print(top_100_genes.head(10))

# Save the Top 100 gene names
top_genes_file = os.path.join(processed_dir, "top_100_predictive_genes.csv")
top_100_genes.to_csv(top_genes_file, index=False)

# Create a slimmed-down expression matrix using ONLY the top 100 genes
# This is what we will feed into our Deep Learning model!
slim_expr_df = expr_df[top_100_genes['Gene_ID']]
slim_expr_file = os.path.join(processed_dir, "GSE65682_expression_TOP100.csv")
slim_expr_df.to_csv(slim_expr_file)

print(f"\n[+] Saved slim matrix (479 patients, 100 genes) -> {slim_expr_file}")
print("=== PHASE 3: FEATURE SELECTION COMPLETE ===")

[*] Loading Processed Data (This will take a moment)...
[+] Data loaded perfectly. 24646 genes ready for evaluation.

[*] Training XGBoost to find the most predictive genes...
[*] Ranking features...

=== THE TOP 10 PREDICTIVE GENES ===
             Gene_ID  Importance
21197  11756520_x_at    0.036323
1641     11717317_at    0.026296
1086   11716595_s_at    0.024438
17249  11748518_a_at    0.024081
12220    11737777_at    0.023168
1653     11717335_at    0.021719
8744   11728776_s_at    0.020346
14982  11744130_a_at    0.019082
1576   11717235_s_at    0.018882
19370  11753640_x_at    0.017916

[+] Saved slim matrix (479 patients, 100 genes) -> /workspace/data/processed/GSE65682_expression_TOP100.csv
=== PHASE 3: FEATURE SELECTION COMPLETE ===


In [1]:
import os
import gzip
import pandas as pd

processed_dir = "/workspace/data/processed"
top_genes_file = os.path.join(processed_dir, "top_100_predictive_genes.csv")
gpl_file = "/workspace/data/raw/GPL13667_family.soft.gz"

print("[*] Loading Top 100 Probes...")
top_100_df = pd.read_csv(top_genes_file)

# Convert our 100 targets into a fast "set" for instantaneous lookup
target_probes = set(top_100_df['Gene_ID'].astype(str).tolist())

print(f"[*] Streaming 6GB Dictionary line-by-line (Memory Safe Mode)...")
print(f"[*] This might take 2-3 minutes, but it will use almost zero RAM.")

translation_dict = {}

# Open the 6GB file without unzipping it into memory
with gzip.open(gpl_file, 'rt') as f:
    in_table = False
    header_cols = []
    
    for line in f:
        line = line.strip()
        
        # Fast-forward until we find the start of the data table
        if line == '!platform_table_begin':
            in_table = True
            continue
        if line == '!platform_table_end':
            break # We are done!
            
        if in_table:
            # The first line of the table contains the column names
            if not header_cols:
                header_cols = line.split('\t')
                id_idx = header_cols.index('ID')
                # Find which column holds the Symbol and Title
                symbol_idx = header_cols.index('Gene Symbol') if 'Gene Symbol' in header_cols else -1
                title_idx = header_cols.index('Gene Title') if 'Gene Title' in header_cols else -1
                continue
            
            # Split the row by tabs
            parts = line.split('\t')
            probe_id = parts[id_idx]
            
            # Is this probe one of our Top 100?
            if probe_id in target_probes:
                symbol = parts[symbol_idx] if symbol_idx != -1 and symbol_idx < len(parts) else 'Unknown'
                title = parts[title_idx] if title_idx != -1 and title_idx < len(parts) else 'Unknown'
                
                # Save it to our tiny dictionary
                translation_dict[probe_id] = {'Gene Symbol': symbol, 'Gene Title': title}
                
                # Print a success message so you know it's working
                print(f"    -> Translated: {probe_id} = {symbol}")
                
                # If we found all 100, we can stop reading the 6GB file entirely!
                if len(translation_dict) == len(target_probes):
                    print("\n[+] Found all 100 genes! Stopping search early to save time.")
                    break

print("\n[*] Mapping complete. Merging with original data...")

# Map the dictionary back to our DataFrame
top_100_df['Gene Symbol'] = top_100_df['Gene_ID'].astype(str).map(
    lambda x: translation_dict.get(x, {}).get('Gene Symbol', 'Unknown/Unmapped')
)
top_100_df['Gene Title'] = top_100_df['Gene_ID'].astype(str).map(
    lambda x: translation_dict.get(x, {}).get('Gene Title', 'Unknown/Unmapped')
)

print("\n=== THE TOP 10 TRANSLATED GENES ===")
print(top_100_df[['Gene Symbol', 'Importance', 'Gene Title']].head(10))

translated_file = os.path.join(processed_dir, "top_100_translated.csv")
top_100_df.to_csv(translated_file, index=False)
print(f"\n[+] Translation complete! Saved to {translated_file}")

[*] Loading Top 100 Probes...
[*] Streaming 6GB Dictionary line-by-line (Memory Safe Mode)...
[*] This might take 2-3 minutes, but it will use almost zero RAM.
    -> Translated: 11715178_s_at = OR2A14
    -> Translated: 11715462_x_at = LYZ
    -> Translated: 11715507_s_at = MAGED1
    -> Translated: 11715533_a_at = LRP1
    -> Translated: 11715806_a_at = SEPW1
    -> Translated: 11715996_a_at = NOSIP
    -> Translated: 11716032_s_at = ARF3
    -> Translated: 11716143_a_at = SIGMAR1
    -> Translated: 11716595_s_at = SLC44A1
    -> Translated: 11716852_x_at = IMPAD1
    -> Translated: 11717235_s_at = RPS7
    -> Translated: 11717317_at = AGPAT6
    -> Translated: 11717335_at = TRIM27
    -> Translated: 11718088_at = TNKS
    -> Translated: 11718375_x_at = WDR5
    -> Translated: 11718818_s_at = SLC26A6
    -> Translated: 11718841_s_at = IL8
    -> Translated: 11718908_s_at = CHST2
    -> Translated: 11719891_s_at = CRYZ
    -> Translated: 11719928_a_at = KDM5B
    -> Translated: 117200

In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score
import os

print("[*] Initializing Phase 4: Multimodal Deep Learning...")

# ---------------------------------------------------------
# 1. LOAD AND MERGE THE DATA
# ---------------------------------------------------------
processed_dir = "/workspace/data/processed"

# Load Clinical (Cleaned) and Genomic (Top 100)
clinical_df = pd.read_csv(os.path.join(processed_dir, "GSE65682_clinical_clean.csv"), index_col=0)
expr_df = pd.read_csv(os.path.join(processed_dir, "GSE65682_expression_TOP100.csv"), index_col=0)

# Merge them together so every row has both clinical and genetic data
full_df = clinical_df.join(expr_df, how='inner')

# ---------------------------------------------------------
# 2. ENCODE CLINICAL VARIABLES
# ---------------------------------------------------------
print("[*] Encoding Clinical Modality...")
le = LabelEncoder()
# Convert text like 'male'/'female' and 'cap'/'no-cap' to 0s and 1s
full_df['gender'] = le.fit_transform(full_df['gender'].astype(str))
full_df['pneumonia diagnoses'] = le.fit_transform(full_df['pneumonia diagnoses'].astype(str))
full_df['diabetes_mellitus'] = le.fit_transform(full_df['diabetes_mellitus'].astype(str))

# Our Clinical Features
clinical_features = ['age', 'gender', 'pneumonia diagnoses', 'diabetes_mellitus']
# Our Genomic Features (The 100 probe IDs)
genomic_features = expr_df.columns.tolist()

X_clinical = full_df[clinical_features].values
X_genomic = full_df[genomic_features].values
y = full_df['mortality_event_28days'].values

# Scale the data (Neural Networks need numbers between -1 and 1)
scaler_clin = StandardScaler()
scaler_gen = StandardScaler()

X_clinical = scaler_clin.fit_transform(X_clinical)
X_genomic = scaler_gen.fit_transform(X_genomic)

# Split into Train (80%) and Test (20%) Sets
Xc_train, Xc_test, Xg_train, Xg_test, y_train, y_test = train_test_split(
    X_clinical, X_genomic, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# 3. DEFINE THE PYTORCH MULTIMODAL NETWORK
# ---------------------------------------------------------
class MultimodalSepsisNet(nn.Module):
    def __init__(self, num_clinical, num_genomic):
        super(MultimodalSepsisNet, self).__init__()
        
        # Branch 1: Clinical Processor (Tiny network for 4 features)
        self.clinical_branch = nn.Sequential(
            nn.Linear(num_clinical, 16),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Branch 2: Genomic Processor (Bigger network for 100 features)
        self.genomic_branch = nn.Sequential(
            nn.Linear(num_genomic, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        
        # Fusion Head: Combines both branches (16 + 32 = 48)
        self.fusion_head = nn.Sequential(
            nn.Linear(48, 24),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(24, 1) # Output: 1 number (Probability of mortality)
        )

    def forward(self, clin_x, gen_x):
        clin_out = self.clinical_branch(clin_x)
        gen_out = self.genomic_branch(gen_x)
        
        # Glue them together!
        fused = torch.cat((clin_out, gen_out), dim=1)
        return self.fusion_head(fused)

# ---------------------------------------------------------
# 4. TRAIN THE AI
# ---------------------------------------------------------
print("[*] Booting up Neural Network...")
model = MultimodalSepsisNet(num_clinical=4, num_genomic=100)

# Convert to PyTorch Tensors
Xc_tr, Xg_tr, y_tr = torch.FloatTensor(Xc_train), torch.FloatTensor(Xg_train), torch.FloatTensor(y_train).view(-1, 1)
Xc_te, Xg_te, y_te = torch.FloatTensor(Xc_test), torch.FloatTensor(Xg_test), torch.FloatTensor(y_test).view(-1, 1)

# Loss and Optimizer
# pos_weight helps with the class imbalance (more survivors than non-survivors)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([3.0]))
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("\n--- BEGIN TRAINING (100 Epochs) ---")
for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    
    # Forward Pass
    predictions = model(Xc_tr, Xg_tr)
    loss = criterion(predictions, y_tr)
    
    # Backward Pass (Learning)
    loss.backward()
    optimizer.step()
    
    # Print progress every 20 epochs
    if (epoch + 1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            test_preds = model(Xc_te, Xg_te)
            test_loss = criterion(test_preds, y_te)
            # Calculate AUROC (The standard medical ML metric)
            probs = torch.sigmoid(test_preds).numpy()
            auroc = roc_auc_score(y_test, probs)
            print(f"Epoch [{epoch+1}/100] | Train Loss: {loss.item():.4f} | Test Loss: {test_loss.item():.4f} | Test AUROC: {auroc:.4f}")

print("\n[+] Multimodal Training Complete!")
# Save the trained AI weights to your hard drive
torch.save(model.state_dict(), "/workspace/data/processed/sepsis_multimodal_model.pth")
print("[+] Model saved to hard drive!")

[*] Initializing Phase 4: Multimodal Deep Learning...
[*] Encoding Clinical Modality...
[*] Booting up Neural Network...

--- BEGIN TRAINING (100 Epochs) ---
Epoch [20/100] | Train Loss: 0.9813 | Test Loss: 0.9955 | Test AUROC: 0.7332
Epoch [40/100] | Train Loss: 0.8179 | Test Loss: 0.8669 | Test AUROC: 0.7826
Epoch [60/100] | Train Loss: 0.5444 | Test Loss: 0.7675 | Test AUROC: 0.8243
Epoch [80/100] | Train Loss: 0.2622 | Test Loss: 1.0280 | Test AUROC: 0.8172
Epoch [100/100] | Train Loss: 0.1176 | Test Loss: 1.6175 | Test AUROC: 0.8035

[+] Multimodal Training Complete!
[+] Model saved to hard drive!
